# Setup

This version is configured to use **Ollama locally** for free.

**Before running the notebook:**
1. Install Ollama.
2. Run `ollama pull llama3.2` once in a terminal.
3. Start the Ollama server with `ollama serve` if it is not already running.


In [1]:
!pip install -q 'crewai[litellm]'

In [2]:
from crewai import Agent, Process, Crew, Task, LLM
from IPython.display import display, Markdown
import os
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

# Optional: keeps CrewAI from falling back to its default model in some setups
#os.environ["MODEL"] = "ollama/llama3.2"


In [3]:
# connectivity check for Ollama (this should work if `ollama serve` is running locally)
import requests

try:
    response = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama connection OK" if response.ok else f"Ollama responded with status {response.status_code}")
except Exception as e:
    print("Could not connect to Ollama at http://localhost:11434")
    print("Make sure Ollama is installed, `ollama pull llama3.2` has been run, and `ollama serve` is running.")
    print(f"Details: {e}")


Ollama connection OK


In [4]:
# Define the local LLM
llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)


# Get the data

*   Interviewer
*   company
*   job position
*   job description



In [5]:
# Store the data
interviewer = input("Enter the name of the interviewer: ")
company = input("Enter the name of the company: ")
job_position = input("Enter the job position: ")
job_description = input("Enter the job description: ")

# Interviewer AI Agent

In [6]:
# Create the interver AI agent
interviewer_agent = Agent(
    role = f" You are {interviewer}, employed at {company}, and hiring for job position {job_position}",
    goal = f"You will help the user user prepare for the job interview for job {job_position} with the following description: {job_description}",
    backstory = """
    Experience
    BEAT81
    Analytics Engineer

    BEAT81 · Full-timeBEAT81 · Full-time
    Apr 2024 - Present · 11 mos
    Berlin, Germany
    diconium
    Full-time · 4 yrs 2 mosFull-time · 4 yrs 2 mos

    Senior Cloud Architect
    Apr 2022 - Mar 2024 · 2 yrs
    Microsoft Azure, DBT and +4 skills

    Data Engineer
    Jan 2021 - Mar 2022 · 1 yr 3 mosJan 2021 to Mar 2022 · 1 yr 3 mos
    DBT, AWS and +4 skills

    Data Scientist
    Feb 2020 - Dec 2020 · 11 mos
    SQL and Python
    4flow logo

    Data Scientist
    4flow · Full-time4flow · Full-time
    Jun 2019 - Dec 2019 · 7 mosJun 2019 to Dec 2019 · 7 mos
    SQL and Python
    DCMN
    Full-time · 3 yrs 3 mos

    Data Scientist
    Apr 2017 - Apr 2019 · 2 yrs 1 moApr 2017 to Apr 2019 · 2 yrs 1 mo
    SQL and Python
    Data Analyst / Business Intelligence Manager

    Feb 2016 - Mar 2017 · 1 yr 2 mosFeb 2016 to Mar 2017 · 1 yr 2 mos
    SQL and Python
    iversity logo

    Data Analyst
    iversity · Full-timeiversity · Full-time
    Dec 2014 - Dec 2015 · 1 yr 1 moDec 2014 to Dec 2015 · 1 yr 1 mo
    SQL and R

    Education
    Technische Universität Berlin
    Master of Science - MS, Computational Neuroscience
    Sep 2009 - Aug 2013

    Universität Osnabrück
    Bachelor of Science - BS, Cognitive ScienceBachelor of Science - BS, Cognitive Science
    """,
    llm = llm
)

In [7]:
# Define the interview prep task
interview_prep_task = Task(
    description = f"""
    Interview the user for the job {job_position} with the following description: {job_description}
    """,
    expected_output = f"""
    ask only one interview question that is relevant for the job {job_position} and has not been asked before.
    return only the question text with no introduction, explanation, or extra commentary.
    """,
    agent = interviewer_agent,
    human_input = False
)

# AI Coach

In [8]:
# Build the AI interview coach agent
coach_agent = Agent(
    role = "Interview Coach",
    goal = f"I help the user prepare for the job interview for job {job_position} by grading the relevance of the answer",
    backstory = "You are an expert in job interviews",
    llm = llm)

In [9]:
# Build the coaching task after the user answers the interview question
def build_coaching_task(question, user_answer):
    return Task(
        description = f"""
        Interview question: {question}

        User answer: {user_answer}

        Evaluate only the user's actual answer.
        If the answer is empty, say clearly that no answer was provided.
        Do not invent strengths if there is no meaningful answer.
        """,
        expected_output = """
        Return the feedback in this format:
        - User answer
        - Good points
        - Weak points
        - How to improve
        If the answer is empty, state that directly.
        """,
        agent = coach_agent
    )

# AI Crew

In [10]:
# Create an empty list for questions
interview_questions = []
# store each question
#interview_questions.append(result.tasks_output[0].raw)
#interview_content = "".join(interview_questions)
#string_source = StringKnowledgeSource(content=interview_content)

In [14]:
# Run the interviewer first
question_crew = Crew(
    agents=[interviewer_agent],
    tasks=[interview_prep_task],
    verbose=False,
    process=Process.sequential,
    memory=False
)

question_result = question_crew.kickoff()
question = question_result.tasks_output[0].raw.strip()

print("Interview question:")
print(question)

user_answer = input("Your answer: ").strip()

coaching_task = build_coaching_task(question, user_answer)

coach_crew = Crew(
    agents=[coach_agent],
    tasks=[coaching_task],
    verbose=False,
    process=Process.sequential,
    memory=False
)

coach_result = coach_crew.kickoff()

Interview question:
Can you walk me through a situation where you encounter conflicting data interpretation for a business problem? How would you systematically resolve this discrepancy to provide accurate analytical insights that support informed decision-making?


In [15]:
# Check output
display(Markdown(f"**Interview question**\n\n{question}"))
print("--------------------")
display(Markdown(f"**Your answer**\n\n{user_answer if user_answer else '_No answer provided_'}"))
print("--------------------")
display(Markdown(coach_result.tasks_output[0].raw))

**Interview question**

Can you walk me through a situation where you encounter conflicting data interpretation for a business problem? How would you systematically resolve this discrepancy to provide accurate analytical insights that support informed decision-making?

--------------------


**Your answer**

I have not encountered conflicting data interpretation

--------------------


- User answer: I have not encountered conflicting data interpretation

- Good points: None

- Weak points: The answer indicates that the user has not personally faced a situation where they had to resolve conflicting data interpretations. This lack of experience may indicate limited exposure to real-world scenarios or critical thinking in resolving such discrepancies.

- How to improve: Consider providing an example from your previous work experiences, academic projects, or personal endeavors where you encountered conflicting data interpretation and how you resolved the issue. Describe the steps you took to identify the discrepancy, evaluate both interpretations, and determine the most accurate approach. This would demonstrate your ability to handle such situations in a professional context as a Data Analytics Working Student.